# Data Cleaning

**Period:** September 4, 2025 → February 25, 2026  
**Source:** Oura Cloud Export

## Objectives
1. Load raw data
2. Identify and remove incomplete days (NaN values)
3. Rename columns for easier manipulation
4. Convert data types (dates, durations)
5. Create derived features (ratios, percentages)
6. Save cleaned dataset

**Expected output:** `data/oura_clean.csv` ready for analysis

In [25]:
# IMPORTS

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Libraries imported successfully!
Pandas: 3.0.1
NumPy: 2.4.2


In [26]:
# LOAD RAW DATA

# Load CSV
df_raw = pd.read_csv('/Users/tim/Projects/custom-readiness-score/data/oura_data.csv')

# Display info
print(f"Data loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"\nAvailable columns:")
print(df_raw.columns.tolist())

Data loaded: 164 rows x 23 columns

Available columns:
['date', 'light_sleep', 'temperature', 'rhr', 'rem_sleep', 'readiness_score', 'sleep_efficiency', 'hrv', 'total_sleep', 'deep_sleep', 'sleep_score', 'total_sleep_min', 'deep_sleep_min', 'rem_sleep_min', 'light_sleep_min', 'total_sleep_h', 'hrv_rhr_ration', 'deep_pct', 'rem_pct', 'day_of_week', 'day_name', 'weekend', 'hrv_rhr_ratio']


In [27]:
# DATA PREVIEW

print("First 5 rows:")
display(df_raw.head())

print("Last 5 rows:")
display(df_raw.tail())

First 5 rows:


,date,light_sleep,temperature,rhr,rem_sleep,readiness_score,sleep_efficiency,hrv,total_sleep,deep_sleep,...,rem_sleep_min,light_sleep_min,total_sleep_h,hrv_rhr_ration,deep_pct,rem_pct,day_of_week,day_name,weekend,hrv_rhr_ratio
0,2025-09-04,12840.0,0.03,55.34,3420.0,73,68.0,66.0,22110.0,5850.0,...,57.0,214.0,6.141667,0.011926,26.458616,15.468114,3,Thursday,False,1.192627
1,2025-09-05,4350.0,1.26,52.11,2460.0,57,92.0,67.0,10800.0,3990.0,...,41.0,72.5,3.000000,0.012857,36.944444,22.777778,4,Friday,False,1.285742
2,2025-09-06,10800.0,0.37,61.27,4140.0,84,80.0,58.0,20490.0,5550.0,...,69.0,180.0,5.691667,0.009466,27.086384,20.204978,5,Saturday,True,0.946630
3,2025-09-07,10320.0,0.35,53.92,4800.0,82,77.0,74.0,21420.0,6300.0,...,80.0,172.0,5.950000,0.013724,29.411765,22.408964,6,Sunday,True,1.372404
4,2025-09-08,10890.0,0.62,71.62,3930.0,37,91.0,39.0,20130.0,5310.0,...,65.5,181.5,5.591667,0.005445,26.378539,19.523100,0,Monday,False,0.544541


Last 5 rows:


,date,light_sleep,temperature,rhr,rem_sleep,readiness_score,sleep_efficiency,hrv,total_sleep,deep_sleep,...,rem_sleep_min,light_sleep_min,total_sleep_h,hrv_rhr_ration,deep_pct,rem_pct,day_of_week,day_name,weekend,hrv_rhr_ratio
159,2026-02-21,10950.0,-0.18,53.11,7710.0,80,93.0,55.0,25320.0,6660.0,...,128.5,182.5,7.033333,0.010356,26.303318,30.450237,5,Saturday,True,1.035587
160,2026-02-22,12600.0,0.31,52.37,8220.0,78,87.0,65.0,27690.0,6870.0,...,137.0,210.0,7.691667,0.012412,24.810401,29.685807,6,Sunday,True,1.241169
161,2026-02-23,14250.0,-0.27,49.79,8820.0,86,89.0,72.0,29280.0,6210.0,...,147.0,237.5,8.133333,0.014461,21.209016,30.122951,0,Monday,False,1.446074
162,2026-02-24,15300.0,-0.16,51.73,8580.0,80,81.0,68.0,31050.0,7170.0,...,143.0,255.0,8.625000,0.013145,23.091787,27.632850,1,Tuesday,False,1.314518
163,2026-02-25,10320.0,-0.33,51.61,4530.0,80,89.0,53.0,20910.0,6060.0,...,75.5,172.0,5.808333,0.010269,28.981349,21.664275,2,Wednesday,False,1.026933


In [28]:
# DETECT MISSING VALUES

print("Missing values analysis:")

# Count NaN per column
missing = df_raw.isnull().sum()

missing_pct = (missing / len(df_raw)) * 100

missing_df = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing': missing.values,
    'Percentage': missing_pct.values
})

# Filter only columns with NaN and sort
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

display(missing_df)

# Count rows with at least one NaN
rows_with_nan = df_raw.isnull().any(axis=1).sum()

print(f"\nTotal rows: {len(df_raw)}")
print(f"Rows with at least 1 NaN: {rows_with_nan}")

Missing values analysis:


,Column,Missing,Percentage



Total rows: 164
Rows with at least 1 NaN: 0


In [29]:
# IDENTIFY INCOMPLETE DAYS

print("Days with missing values:")

# Filter rows with NaN
incomplete_days = df_raw[df_raw.isnull().any(axis=1)]

# Display full rows
display(incomplete_days)

# Display dates
print(f"\nDates of incomplete days:")
print(incomplete_days['date'].tolist())

# Count NaN per row
print(f"\nNumber of NaN per incomplete day:")
nan_per_row = incomplete_days.isnull().sum(axis=1)
for idx, count in nan_per_row.items():
    date = incomplete_days.loc[idx, 'date']
    print(f"  Row {idx}: {date} → {count} NaN values")

Days with missing values:


,date,light_sleep,temperature,rhr,rem_sleep,readiness_score,sleep_efficiency,hrv,total_sleep,deep_sleep,...,rem_sleep_min,light_sleep_min,total_sleep_h,hrv_rhr_ration,deep_pct,rem_pct,day_of_week,day_name,weekend,hrv_rhr_ratio



Dates of incomplete days:
[]

Number of NaN per incomplete day:


In [30]:
# REMOVE INCOMPLETE DAYS

print(f"Before cleaning: {len(df_raw)} days")

# Remove rows with any NaN
df_clean = df_raw.dropna()

print(f"After cleaning: {len(df_clean)} days")
print(f"Removed: {len(df_raw) - len(df_clean)} days")

# Verify no NaN remaining
nan_remaining = df_clean.isnull().sum().sum()
print(f"\nNaN remaining: {nan_remaining}")

if nan_remaining == 0:
    print("Dataset is clean! No missing values.")
else:
    print(f"Warning: {nan_remaining} NaN still present.")

Before cleaning: 164 days
After cleaning: 164 days
Removed: 0 days

NaN remaining: 0
Dataset is clean! No missing values.


In [31]:
# RENAME COLUMNS 

df_clean.rename(columns={
    'Average HRV' : 'hrv',
    'Average Resting Heart Rate' : 'rhr',
    'Temperature Deviation (°C)' : 'temperature',
    'Total Sleep Duration' : 'total_sleep',
    'Deep Sleep Duration' : 'deep_sleep',
    'REM Sleep Duration' : 'rem_sleep',
    'Light Sleep Duration' : 'light_sleep',
    'Sleep Efficiency' : 'sleep_efficiency',
    'Sleep Score' : 'sleep_score',
    'Readiness Score' : 'readiness_score',

}, inplace=True)

print("\nNew column names:")
print(df_clean.columns.tolist())


New column names:
['date', 'light_sleep', 'temperature', 'rhr', 'rem_sleep', 'readiness_score', 'sleep_efficiency', 'hrv', 'total_sleep', 'deep_sleep', 'sleep_score', 'total_sleep_min', 'deep_sleep_min', 'rem_sleep_min', 'light_sleep_min', 'total_sleep_h', 'hrv_rhr_ration', 'deep_pct', 'rem_pct', 'day_of_week', 'day_name', 'weekend', 'hrv_rhr_ratio']


In [32]:
# CONVERT DATA TYPES

# Converting date to datetime
df_clean['date'] = pd.to_datetime(df_clean['date'])
print(f"Date column type: {df_clean['date'].dtype}")

# Create duration columns in minutes
df_clean['total_sleep_min'] = df_clean['total_sleep'] / 60
df_clean['deep_sleep_min'] = df_clean['deep_sleep'] / 60
df_clean['rem_sleep_min'] = df_clean['rem_sleep'] / 60
df_clean['light_sleep_min'] = df_clean['light_sleep'] / 60

# Create total sleep in hours
df_clean['total_sleep_h'] = df_clean['total_sleep'] / 3600

print("\nNew duration columns:")
display(df_clean[['date', 'total_sleep_h', 'total_sleep_min', 'deep_sleep_min', 'rem_sleep_min', 'light_sleep_min']].head())

Date column type: datetime64[us]

New duration columns:


,date,total_sleep_h,total_sleep_min,deep_sleep_min,rem_sleep_min,light_sleep_min
0,2025-09-04,6.141667,368.5,97.5,57.0,214.0
1,2025-09-05,3.000000,180.0,66.5,41.0,72.5
2,2025-09-06,5.691667,341.5,92.5,69.0,180.0
3,2025-09-07,5.950000,357.0,105.0,80.0,172.0
4,2025-09-08,5.591667,335.5,88.5,65.5,181.5


In [33]:
# CREATE DERIVED FEATURES

#Recovery Ratio
df_clean['hrv_rhr_ratio'] = df_clean['hrv'] / df_clean['rhr']

# Sleep Stage Percentage
df_clean['deep_pct'] = (df_clean['deep_sleep_min'] / df_clean['total_sleep_min']) * 100
df_clean['rem_pct'] = (df_clean['rem_sleep_min'] / df_clean['total_sleep_min']) * 100

# Temporal Features
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
df_clean['day_name'] = df_clean['date'].dt.day_name()
df_clean['weekend'] = df_clean['day_of_week'].isin([5, 6])

print("New features:")
display(df_clean[['hrv_rhr_ratio', 'deep_pct', 'rem_pct', 'day_of_week', 'day_name', 'weekend']].head(10))

New features:


,hrv_rhr_ratio,deep_pct,rem_pct,day_of_week,day_name,weekend
0,1.192627,26.458616,15.468114,3,Thursday,False
1,1.285742,36.944444,22.777778,4,Friday,False
2,0.946630,27.086384,20.204978,5,Saturday,True
3,1.372404,29.411765,22.408964,6,Sunday,True
4,0.544541,26.378539,19.523100,0,Monday,False
5,1.071753,33.484163,26.395173,1,Tuesday,False
6,1.063638,24.518201,26.231263,2,Wednesday,False
7,0.522724,21.189591,19.330855,3,Thursday,False
8,1.204139,31.053604,21.256932,4,Friday,False
9,1.076813,30.937500,25.312500,5,Saturday,True


In [34]:
# SAVE CLEANED DATASET

print("Date is already set as index")
print(f"  Index type: {df_clean.index.dtype}")

if 'hrv_rhr_ration' in df_clean.columns:
    df_clean = df_clean.drop('hrv_rhr_ration', axis=1)
    print("   ✅ Removed 'hrv_rhr_ration' (typo from previous run)")

# Save to CSV (with date as index)
df_clean.to_csv('/Users/tim/Projects/custom-readiness-score/data/oura_clean.csv')

print(f"Cleaned data saved to: data/oura_clean.csv")
print(f"Final dataset: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")

print("\nFinal dataset (with date index):")
display(df_clean[['hrv', 'rhr', 'total_sleep_h', 'readiness_score']].head())

Date is already set as index
  Index type: int64
   ✅ Removed 'hrv_rhr_ration' (typo from previous run)
Cleaned data saved to: data/oura_clean.csv
Final dataset: 164 rows x 22 columns

Final dataset (with date index):


,hrv,rhr,total_sleep_h,readiness_score
0,66.0,55.34,6.141667,73
1,67.0,52.11,3.000000,57
2,58.0,61.27,5.691667,84
3,74.0,53.92,5.950000,82
4,39.0,71.62,5.591667,37
